# RAG from scratch

Pipeline: **load** → chunk → embed → store → retrieve → generate

## Step 1: Load
Goal: turn each PDF into plain text, and keep track of **where the text came from** (file + page).
That "where" is called **metadata**. We need it later to cite sources and to check retrieval against `eval/questions.json`.

In [3]:
from pathlib import Path
from pypdf import PdfReader

DATA_DIR = Path("../data/raw")  # notebooks run fraom the notebooks/ folder

def load_pdfs(folder):
    """Return one record per PDF page: {text, source, page}."""
    pages = []
    for pdf_path in sorted(folder.glob("*.pdf")):
        reader = PdfReader(pdf_path)
        for page_num, page in enumerate(reader.pages, start=1):
            text = page.extract_text() or ""  # scanned/image pages return None
            pages.append({"text": text, "source": pdf_path.name, "page": page_num})
    return pages

pages = load_pdfs(DATA_DIR)
print(f"Loaded {len(pages)} pages")
for p in pages:
    print(f"{p['source']:40} page {p['page']}  {len(p['text']):5} chars")

Loaded 9 pages
01_caching_and_cdn.pdf                   page 1   2475 chars
01_caching_and_cdn.pdf                   page 2   2956 chars
01_caching_and_cdn.pdf                   page 3   1811 chars
02_project_falcon_databases.pdf          page 1   2578 chars
02_project_falcon_databases.pdf          page 2   2761 chars
02_project_falcon_databases.pdf          page 3   1322 chars
03_shopkart_checkout_case_study.pdf      page 1   2704 chars
03_shopkart_checkout_case_study.pdf      page 2   2308 chars
03_shopkart_checkout_case_study.pdf      page 3    306 chars


### Inspect the raw text
Always look at what the loader produced. Check: did the **table** survive? Are there broken words, extra line breaks, or headers/footers repeated on every page?

In [4]:
print(pages[0]["text"][:3000])

Caching and CDN Design Guide
Halcyon Labs Platform Engineering · Internal standard PE-CACHE-07 · Revision 3
1. Why we cache
Caching is the practice of storing the result of an expensive operation so that later requests can be
served faster and more cheaply. At Halcyon Labs almost every read path passes through at least
one cache layer before it touches a database. A well-designed cache reduces latency, protects
downstream systems from load spikes and lowers infrastructure cost. A badly designed cache
serves stale data, hides bugs and fails in surprising ways during incidents.
This guide explains the general principles first and then describes the specific defaults that Halcyon
Labs teams must follow. Where this document says must, a deviation needs approval from the
Platform Architecture Review, which meets every second Thursday.
2. Cache placement and tiers
Caches can live in several places: in the client (browser or mobile app), at the edge (a CDN), in a
shared distributed cache such

## Step 2: Chunk
Goal: split each page into small pieces (**chunks**) that each hold one idea.

Why not embed a whole page?
- One vector for a whole page blurs many topics together, so search gets less precise.
- The LLM prompt gets filled with irrelevant text.
- Embedding models have a max input (bge-small: 512 tokens); longer text gets cut off.

Strategy here: **fixed-size word chunks with overlap** (the simplest one).
- `chunk_size=150` words (~200 tokens). Our docs are small, so small chunks make retrieval actually work.
- `overlap=30` words: neighbouring chunks share some words, so a sentence cut at a boundary still appears whole in one of them.

In [5]:
def chunk_text(text, chunk_size=150, overlap=30):
    """Split text into chunks of `chunk_size` words; neighbours share `overlap` words."""
    words = text.split()            # split on any whitespace -> list of words
    step = chunk_size - overlap     # how far to move forward each time (120)
    chunks = []
    for start in range(0, len(words), step):   # start = 0, 120, 240, ...
        chunks.append(" ".join(words[start:start + chunk_size]))
        if start + chunk_size >= len(words):   # this chunk reached the end
            break
    return chunks

# Try it on a tiny example first: 10 "words", chunks of 4, overlap 1
print(chunk_text("w1 w2 w3 w4 w5 w6 w7 w8 w9 w10", chunk_size=4, overlap=1))

['w1 w2 w3 w4', 'w4 w5 w6 w7', 'w7 w8 w9 w10']


In [6]:
chunks = []
for p in pages:
    for i, text in enumerate(chunk_text(p["text"])):
        chunks.append({
            "id": f"{p['source']}-p{p['page']}-c{i}",  # unique ID, e.g. 01_caching_and_cdn.pdf-p1-c0
            "text": text,
            "source": p["source"],  # metadata copied from the page
            "page": p["page"],
        })

print(f"{len(chunks)} chunks")
print("words per chunk:", [len(c["text"].split()) for c in chunks])

30 chunks
words per chunk: [150, 150, 150, 55, 150, 150, 150, 133, 150, 150, 55, 150, 150, 150, 35, 150, 150, 150, 101, 150, 96, 150, 150, 150, 71, 150, 150, 150, 31, 50]


In [7]:
# Check the overlap: the end of chunk 0 should repeat at the start of chunk 1
print("END OF CHUNK 0:  ...", chunks[0]["text"][-150:])
print()
print("START OF CHUNK 1:", chunks[1]["text"][:150], "...")

END OF CHUNK 0:  ... e Review, which meets every second Thursday. 2. Cache placement and tiers Caches can live in several places: in the client (browser or mobile app), at

START OF CHUNK 1: approval from the Platform Architecture Review, which meets every second Thursday. 2. Cache placement and tiers Caches can live in several places: in  ...
